# OPERA-LM chat demo -- full pipeline (Colab GPU)

Runs the whole chain end to end:
1. Clone the repo (opera_lm library + opera-chat scripts, single GitHub repo)
2. Prepare the **full** smoltalk dataset (no 150k-conversation cap -- that cap existed only to protect a 16GB laptop, not needed here)
3. Train a production-scale ~155M-param model (d=1664, nb=416, 8 layers) with the Muon-routing fix, sized up from the README's 20M-param laptop config now that we have real GPU headroom
4. Short state-passing fine-tune phase (~500 steps) for length generalization
5. Push the result to a Hugging Face Space so people can try it

Runtime > Change runtime type > GPU (T4 is fine; A100 is faster if you have Colab Pro) before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


## 1. Clone + install

In [ ]:
REPO_URL = "https://github.com/Merna-Khalid/OPERA-LM"

!git clone $REPO_URL repo
%cd repo
!pip install -q datasets tokenizers "gradio>=6" huggingface_hub

Cloning into 'repo'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 67 (delta 18), reused 65 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 359.65 KiB | 370.00 KiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/repo/repo


## 2. (Optional) mount Drive for checkpoint persistence
Free Colab sessions can disconnect. Checkpoints under `OUT_ROOT` on Drive survive a disconnect; `--resume` picks back up from the last `--save-every` checkpoint. Skip this cell to keep everything local to the (ephemeral) Colab VM instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = '/content/drive/MyDrive/opera-lm-demo'
import os; os.makedirs(OUT_ROOT, exist_ok=True)

Mounted at /content/drive


In [ ]:
# If you skipped the Drive cell above, uncomment this instead:
# OUT_ROOT = '/content/opera-lm-demo'
# import os; os.makedirs(OUT_ROOT, exist_ok=True)
print(OUT_ROOT)

## 3. Prepare the full dataset
Drops the 150k-conversation cap: `--n-convs` set far above the real size of `HuggingFaceTB/smoltalk` ("all" split) so the loop simply exhausts the dataset instead of stopping early. `--eval-max-len 2048` (vs the original 1024) so held-out `test_long` conversations give us real extrapolation signal past the state-passing phase's 1024-token cap too, not just up to it.

Reuses the existing `tokenizer.json` already in the repo (trained on 500k lines from this same dataset -- no need to retrain for more conversations from the same distribution).

In [ ]:
DATA_FULL = f'{OUT_ROOT}/data_chat_full.pkl'
!python opera-chat/prepare_data.py \
  --tokenizer opera-chat/tokenizer.json \
  --out $DATA_FULL \
  --n-convs 5000000 \
  --max-len 256 \
  --eval-max-len 2048

README.md: 100% 9.72k/9.72k [00:00<00:00, 24.5MB/s]

data/all/train-00000-of-00009.parquet: downloading bytes:  97% 217M/223M [00:02<00:00, 185MB/s, 18.6MB/s  ]
data/all/train-00000-of-00009.parquet: downloading bytes: 100% 222M/222M [00:02<00:00, 82.8MB/s, 20.4MB/s  ]
data/all/train-00000-of-00009.parquet: reconstructing file: 100% 223M/223M [00:02<00:00, 82.9MB/s, 21.0MB/s  ]

data/all/train-00001-of-00009.parquet: downloading bytes:  77% 173M/223M [00:02<00:00, 174MB/s, 11.3MB/s  ]
data/all/train-00001-of-00009.parquet: reconstructing file:  30% 67.1M/223M [00:02<00:05, 29.7MB/s]
data/all/train-00001-of-00009.parquet: downloading bytes: 100% 223M/223M [00:02<00:00, 196MB/s, 19.7MB/s  ]
data/all/train-00001-of-00009.parquet: downloading bytes: 100% 223M/223M [00:02<00:00, 89.2MB/s, 20.7MB/s  ]
data/all/train-00001-of-00009.parquet: reconstructing file: 100% 223M/223M [00:02<00:00, 89.3MB/s, 20.9MB/s  ]

data/all/train-00002-of-00009.parquet: downloading bytes: 100% 223M/223M [00:02<0

## 4. Full production training run -- scaled to ~155M params
The README's reference config (d=640, nb=160, 4 layers, ~20M params) was sized for a 16GB laptop. Colab GPUs have real headroom, so this uses **d=1664, nb=416, num_layers=8 (~155M params, confirmed by instantiating the model directly -- `d` must equal `4*nb`, an architectural constraint in `model.py`)**. Same OPERA identity as always: `pe_mode=none` / `fold_mode=left` / `rot_mode=free`, Muon optimizer (with the routing fix -- `mem_beta` now correctly goes to AdamW).

**On CUDA optimization specifically** (no `--metal` flag here -- that fused kernel is Apple Silicon/MPS-only): `opera_lm/train.py` already gates AMP dtype on GPU compute capability (bf16 only on Ampere+/sm_80, e.g. A100; fp16+GradScaler on Turing, e.g. T4, since Turing only *emulates* bf16 and that both breaks `torch.compile`'s Inductor graphs and runs slow), enables TF32 for full-precision matmuls, and -- unlike MPS, where `torch.compile` is gated to `fold_mode='scan'` only -- **enables `torch.compile` unconditionally on CUDA regardless of fold mode**, so `fold_mode='left'` gets real Inductor/Triton fusion here that it never got on MPS. This is all pre-existing, verified by reading the code, not new.

**Smoke test first**: 155M params on data at full scale, on a GPU tier we haven't measured yet (T4 vs A100 makes a large difference), is not something to commit 20k steps to blind. Run ~30 steps, read the `s/step` figure it prints, and use that to sanity-check total wall-clock and pick `--batch` before launching the real run below.

In [ ]:
!python opera-chat/train_chat.py \
  --data $DATA_FULL \
  --steps 30 \
  --batch 16 \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --compile default \
  --out-dir /content/smoke_full \
  --save-every 0

data: train=223987 short=11692 long=44614 vocab=16384
=== OPERA-LM v9.0 (Spinor Tree, tree-training line) [pe-none+data-chat+rotfree+amp+foreach+cmp-default+gpudata+aux0.25+gb2.0,0.0,-2.0+cur64x250+tie+opt-muon+msup] ===
  steps=30, batch=16, train max_len=256, eval_max_len=2048
  d=1664, nb=416, num_layers=8, lock=none, device=cuda
  pe=none, fold=left, fold_rotors=shared, fold_scale=False
  norm=layer, act=tanh, node_residual=False
  rot=free, seed=42, data=chat
  out=/content/smoke_full, save_every=0, resume=False
  tie=True, dropout=0.0, msup=True (weight 0.1)
  OPT: compile=default, gpu_data=True, aux_frac=0.25, amp=True, foreach=True
  References (train<=20, 4L): OPERA pe-none 70.92 / 1.65x / 1.68x; RoPE transformer 70.71 / 1.60x / 1.68x
  Fold work @T=256: 769 row-composes/layer (v7.7 masked: 1792; 2.33x less)
  Model params: 154,815,249
  torch.compile enabled (mode=default, recompile_limit=64)
  GPU batch source: 223,987 sequences on cuda (437 MiB)
  Muon: 88,694,528 matrix pa

In [ ]:
BATCH = 16  # set from the smoke test above
RUN_BASE = f'{OUT_ROOT}/runs_chat_full'
!python opera-chat/train_chat.py \
  --data $DATA_FULL \
  --steps 20000 \
  --batch $BATCH \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --compile default \
  --out-dir $RUN_BASE \
  --save-every 1000 \
  --resume

data: train=223987 short=11692 long=44614 vocab=16384
=== OPERA-LM v9.0 (Spinor Tree, tree-training line) [pe-none+data-chat+rotfree+amp+foreach+cmp-default+gpudata+aux0.25+gb2.0,0.0,-2.0+cur64x250+tie+opt-muon+msup] ===
  steps=20000, batch=16, train max_len=256, eval_max_len=2048
  d=1664, nb=416, num_layers=8, lock=none, device=cuda
  pe=none, fold=left, fold_rotors=shared, fold_scale=False
  norm=layer, act=tanh, node_residual=False
  rot=free, seed=42, data=chat
  out=/content/drive/MyDrive/opera-lm-demo/runs_chat_full, save_every=1000, resume=True
  tie=True, dropout=0.0, msup=True (weight 0.1)
  OPT: compile=default, gpu_data=True, aux_frac=0.25, amp=True, foreach=True
  References (train<=20, 4L): OPERA pe-none 70.92 / 1.65x / 1.68x; RoPE transformer 70.71 / 1.60x / 1.68x
  Fold work @T=256: 769 row-composes/layer (v7.7 masked: 1792; 2.33x less)
  Model params: 154,815,249
  torch.compile enabled (mode=default, recompile_limit=64)
  GPU batch source: 223,987 sequences on cuda (

## 5. State-passing fine-tune (length generalization)
Buitrago Ruiz & Gu, ICML 2025 (arXiv:2507.02782): seed the compose function with genuinely deep recursion states by training briefly on chains of concatenated real sequences, rather than only ever seeing states reachable within a single training-length sequence. This is a short (~500-step, ~0.1% of the pretraining budget) fine-tune from the base checkpoint above, not a new architecture or a full retrain -- and it validated positively at toy scale earlier (state-passing beat baseline in every extrapolation bucket, gap growing with distance from training length).

In [ ]:
import glob, os
base_ckpts = [f for f in glob.glob(f'{RUN_BASE}/*.pt') if not f.endswith('_train_ckpt.pt')]
BASE_CKPT = max(base_ckpts, key=os.path.getmtime)
print('base checkpoint:', BASE_CKPT)

DATA_SP = f'{OUT_ROOT}/data_chat_full_statepassing.pkl'
!python opera-chat/prep_statepassing_data.py \
  --src $DATA_FULL --dst $DATA_SP \
  --n-synth 50000 --max-len 1024

In [ ]:
RUN_SP = f'{OUT_ROOT}/runs_chat_sp'
!python opera-chat/train_chat.py \
  --data $DATA_SP \
  --init-weights-from $BASE_CKPT \
  --steps 500 \
  --batch 8 \
  --max-len 1024 \
  --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --no-curriculum \
  --compile default \
  --out-dir $RUN_SP \
  --save-every 0

usage: train_chat.py [-h] [--data DATA] [--tokenizer TOKENIZER]
                     [--steps STEPS] [--batch BATCH] [--max-len MAX_LEN]
                     [--eval-max-len EVAL_MAX_LEN] [--d D] [--nb NB]
                     [--num-layers NUM_LAYERS] [--device DEVICE]
                     [--out-dir OUT_DIR] [--save-every SAVE_EVERY] [--resume]
                     [--max-lr MAX_LR] [--warmup-steps WARMUP_STEPS]
                     [--seed SEED] [--no-msup] [--no-curriculum]
                     [--curriculum-t0 CURRICULUM_T0]
                     [--curriculum-every CURRICULUM_EVERY]
                     [--optimizer {adamw,muon}] [--muon-lr MUON_LR]
                     [--readout {none,multistate}] [--mem {none,delta}]
                     [--mem-dim MEM_DIM]
                     [--compile {default,reduce-overhead,max-autotune,off}]
                     [--metal] [--init-weights-from INIT_WEIGHTS_FROM]
train_chat.py: error: argument --data: expected one argument


## 6. Quick local check before publishing
Launches a temporary Gradio share link (dies when this Colab session ends) so you can sanity-check the model right here before deciding whether it's worth publishing.

In [ ]:
import os
os.environ['MODEL_DIR'] = RUN_SP
cfg_path = f'{RUN_SP}/model_config.json'
assert os.path.exists(cfg_path), f'missing {cfg_path}'
%cd opera-chat
!MODEL_DIR=$RUN_SP python -c "import app; app.demo.launch(share=True)"
%cd ..

AssertionError: missing /content/drive/MyDrive/opera-lm-demo/runs_chat_sp/model_config.json

## 7. Publish to a Hugging Face Space
Set `HF_TOKEN` as a Colab secret first (key icon in the left sidebar) with **write** access, then fill in your HF username below. This creates/updates a model repo (checkpoint + config + tokenizer) and a Gradio Space repo (self-contained, pulls the model from the model repo automatically -- no secrets needed in the Space itself).

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

sp_ckpts = [f for f in glob.glob(f'{RUN_SP}/*.pt') if not f.endswith('_train_ckpt.pt')]
SP_CKPT = max(sp_ckpts, key=os.path.getmtime)
print('SP checkpoint:', SP_CKPT)

HF_USERNAME = 'YOUR_HF_USERNAME'  # <-- fill in
MODEL_REPO = f'{HF_USERNAME}/opera-lm-chat'
SPACE_REPO = f'{HF_USERNAME}/opera-lm-chat-space'

!python opera-chat/push_to_hub.py \
  --ckpt $SP_CKPT \
  --config $RUN_SP/model_config.json \
  --tokenizer opera-chat/tokenizer.json \
  --model-repo $MODEL_REPO \
  --space-repo $SPACE_REPO

## 8. Matched transformer baseline (for comparison)
Same protocol as the base OPERA run: same `DATA_FULL` pkl (identical BPE tokenization), same `--steps`/`--max-len`/`--eval-max-len`, same Muon optimizer, same curriculum. Config sized to match OPERA's ~154.8M params as closely as the transformer's own shape constraints (`nheads` must divide `d`) allow: **d=1264, nheads=8, num_layers=7 -> 155.05M params (+0.16% vs OPERA's 154.8M)**, confirmed the same way (instantiating `TransformerBaseline` directly, see below).

`--pe rope` (rotary) is the standard choice for this comparison -- `nope`/`sin`/`learned` are also available if you want more points of comparison later. Known asymmetry, not a bug: `train_tf_chat.py` has no `--compile` flag (this baseline always runs eager) -- matches this project's existing convention of matching data/protocol while leaving architecture-native optimizations (OPERA's `torch.compile`, Muon-routing, etc.) as each model's own.

Two bugs fixed in these files while wiring this up (same classes of bug as elsewhere in this session):
- `opera_transformer_baseline_v2.py` imported 7 names from a legacy file (`opera_v8_spinor_optimized.py`) that isn't part of this repo; only `sinusoidal_pos_enc` (used when `--pe sin`) was actually needed by the `TransformerBaseline` class itself (the rest were dead weight from this file's own unused standalone `main()`), so that's now a small self-contained function instead of an external dependency that would have crashed on import.
- The same unguarded-`compute_perplexity`/no-OOM-retry bug fixed in `opera_lm/train.py` earlier was independently duplicated in `train_tf_chat.py` (it has its own eval calls); now wired through the same `_oom_backstop`/`_eval_batch` helpers.

In [ ]:
!python opera-chat/train_tf_chat.py \
  --pe rope \
  --data $DATA_FULL \
  --steps 30 \
  --batch 16 \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1264 --num-layers 7 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --out-dir /content/smoke_tf \
  --save-every 0

No `torch.compile` here (see above), so `s/step` from this smoke test is directly usable -- no compile-storm startup cost to discount. Adjust `TF_BATCH` accordingly.

In [ ]:
TF_BATCH = 16  # set from the smoke test above
RUN_TF = f'{OUT_ROOT}/runs_chat_tf'
!python opera-chat/train_tf_chat.py \
  --pe rope \
  --data $DATA_FULL \
  --steps 20000 \
  --batch $TF_BATCH \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1264 --num-layers 7 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --out-dir $RUN_TF \
  --save-every 1000 \
  --resume

[baseline 2026-07-19-t3] opera_transformer_baseline_v2.py
data: train=223987 short=11692 long=44614 vocab=16384
=== transformer baseline [rope] on chat protocol ===
  params: 155,049,777 (tied)  steps=20000 batch=16 max_len=256 device=cuda
  Muon: 134,206,464 matrix params (lr 0.02) + AdamW: 20,843,313 (lr 0.001)
  Initial per-token perplexity: 16835464928155.20 (chance ~ 16384)
  step     0  loss 29.9486  lr 0.00000  (0.4s, 0.36s/step)  T_cur 64
  step   200  loss 4.7025  lr 0.00040  (11.2s, 0.06s/step)  T_cur 64
  step   400  loss 4.0469  lr 0.00080  (23.4s, 0.06s/step)  T_cur 128
  step   600  loss 4.4318  lr 0.00100  (37.8s, 0.06s/step)  T_cur 256
  step   800  loss 3.8354  lr 0.00100  (54.1s, 0.07s/step)  T_cur 256
  step  1000  loss 3.9772  lr 0.00100  (70.4s, 0.07s/step)  T_cur 256
    checkpoint -> /content/drive/MyDrive/opera-lm-demo/runs_chat_tf/tf_rope_opt-muon_train_ckpt.pt
    per-token perplexity: 54.53
  step  1200  loss 3.6542  lr 0.00100  (89.5s, 0.07s/step)  T_cur 256

## Stage 2: fresh model -- raw-text pretrain, then smoltalk fine-tune
Separate from everything above (sections 3-8 train a fresh model directly on smoltalk only; this stage is a **different fresh model**, not a continuation of that one). Two phases:
1. Pretrain from scratch on a slice of **FineWeb-Edu** (`HuggingFaceFW/fineweb-edu`, `sample-10BT` config -- educational-filtered web text, the standard small-model pretraining corpus; same one used in Karpathy's `build-nanogpt`), so the model sees broad general text before ever seeing a chat turn.
2. Continue that checkpoint into smoltalk SFT via `--init-weights-from` (same mechanism the state-passing fine-tune already uses) -- this is the "use smoltalk for fine tuning" step.

Same `tokenizer.json` (16384-vocab BPE) for both phases, on purpose: continuing phase 1's checkpoint into phase 2 only works if both share one embedding table. Tradeoff, stated plainly: this tokenizer was built from smoltalk chat text, so it likely segments FineWeb-Edu's more formal prose a bit less efficiently than a tokenizer trained on general web text would -- acceptable to keep the pretrain -> SFT continuation intact, not a hidden flaw.

**Compute budget is an explicit choice, not a silent default**: full Chinchilla-optimal pretraining for a 155M-param model would want roughly 20x params in tokens (~3B tokens) -- multiple times the wall-clock of everything else in this notebook combined. `prepare_fineweb.py`'s default (`--max-tokens 500000000`, 500M tokens) and the pretraining cell's `--steps 20000` below are a deliberately modest slice, matched to a single-session budget, not a claim that this is compute-optimal. Raise both if you want to spend more.

In [ ]:
DATA_FINEWEB = f'{OUT_ROOT}/data_fineweb.pkl'
!python opera-chat/prepare_fineweb.py \
  --tokenizer opera-chat/tokenizer.json \
  --out $DATA_FINEWEB \
  --max-tokens 500000000 \
  --max-len 256 \
  --eval-max-len 2048

README.md: 100% 26.4k/26.4k [00:00<00:00, 61.3MB/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 64984.07it/s]
  5000 docs, ~5,915,087 tokens seen: train=1180 short=75 long=225
  10000 docs, ~11,699,970 tokens seen: train=2421 short=133 long=463
  15000 docs, ~17,303,976 tokens seen: train=3491 short=213 long=730
  20000 docs, ~23,397,795 tokens seen: train=4538 short=268 long=1021
  25000 docs, ~30,160,893 tokens seen: train=5934 short=313 long=1338
  30000 docs, ~36,159,391 tokens seen: train=7268 short=366 long=1584
  35000 docs, ~41,553,000 tokens seen: train=8342 short=429 long=1841
  40000 docs, ~47,212,539 tokens seen: train=9483 short=463 long=2084
  45000 docs, ~53,244,973 tokens seen: train=10496 short=508 long=2389
  50000 docs, ~59,474,753 tokens seen: train=11676 short=573 long=2676
  55000 docs, ~65,403,518 tokens seen: train=13042 short=608 long=2948
  60000 docs, ~71,313,037 tokens seen: train=14303 short=686 long=3257
  65000 docs, ~77,069,202 tokens seen: train=

In [ ]:
!python opera-chat/train_chat.py \
  --data $DATA_FINEWEB \
  --steps 30 \
  --batch 16 \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --compile default \
  --out-dir /content/smoke_fw \
  --save-every 0

data: train=97560 short=5411 long=23526 vocab=16384
=== OPERA-LM v9.0 (Spinor Tree, tree-training line) [pe-none+data-chat+rotfree+amp+foreach+cmp-default+gpudata+aux0.25+gb2.0,0.0,-2.0+cur64x250+tie+opt-muon+msup] ===
  steps=30, batch=16, train max_len=256, eval_max_len=2048
  d=1664, nb=416, num_layers=8, lock=none, device=cuda
  pe=none, fold=left, fold_rotors=shared, fold_scale=False
  norm=layer, act=tanh, node_residual=False
  rot=free, seed=42, data=chat
  out=/content/smoke_fw, save_every=0, resume=False
  tie=True, dropout=0.0, msup=True (weight 0.1)
  OPT: compile=default, gpu_data=True, aux_frac=0.25, amp=True, foreach=True
  References (train<=20, 4L): OPERA pe-none 70.92 / 1.65x / 1.68x; RoPE transformer 70.71 / 1.60x / 1.68x
  Fold work @T=256: 769 row-composes/layer (v7.7 masked: 1792; 2.33x less)
  Model params: 154,815,249
  torch.compile enabled (mode=default, recompile_limit=64)
  GPU batch source: 97,560 sequences on cuda (191 MiB)
  Muon: 88,694,528 matrix params 

In [ ]:
FW_BATCH = 16  # set from the smoke test above
RUN_FW = f'{OUT_ROOT}/runs_fineweb'
!python opera-chat/train_chat.py \
  --data $DATA_FINEWEB \
  --steps 20000 \
  --batch $FW_BATCH \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --compile default \
  --out-dir $RUN_FW \
  --save-every 1000 \
  --resume

data: train=97560 short=5411 long=23526 vocab=16384
=== OPERA-LM v9.0 (Spinor Tree, tree-training line) [pe-none+data-chat+rotfree+amp+foreach+cmp-default+gpudata+aux0.25+gb2.0,0.0,-2.0+cur64x250+tie+opt-muon+msup] ===
  steps=20000, batch=16, train max_len=256, eval_max_len=2048
  d=1664, nb=416, num_layers=8, lock=none, device=cuda
  pe=none, fold=left, fold_rotors=shared, fold_scale=False
  norm=layer, act=tanh, node_residual=False
  rot=free, seed=42, data=chat
  out=/content/drive/MyDrive/opera-lm-demo/runs_fineweb, save_every=1000, resume=True
  tie=True, dropout=0.0, msup=True (weight 0.1)
  OPT: compile=default, gpu_data=True, aux_frac=0.25, amp=True, foreach=True
  References (train<=20, 4L): OPERA pe-none 70.92 / 1.65x / 1.68x; RoPE transformer 70.71 / 1.60x / 1.68x
  Fold work @T=256: 769 row-composes/layer (v7.7 masked: 1792; 2.33x less)
  Model params: 154,815,249
  torch.compile enabled (mode=default, recompile_limit=64)
  GPU batch source: 97,560 sequences on cuda (191 M

### Fine-tune the pretrained checkpoint on smoltalk
This is the "use smoltalk for fine tuning" step -- `--init-weights-from` loads phase 1's pretrained weights, then trains on the same `DATA_FULL` smoltalk pkl from section 3 (reused, not rebuilt). SFT is conventionally a shorter phase than pretraining (a handful of passes over the instruction data, not a from-scratch budget): default here is `--steps 10000`, about half the pretraining budget -- an explicit choice, adjust freely. `--d`/`--nb`/`--num-layers` must match phase 1 exactly (loading a state_dict requires identical shapes).

In [ ]:
import glob

fw_ckpts = [f for f in glob.glob(f'{RUN_FW}/*.pt') if not f.endswith('_train_ckpt.pt')]
FW_CKPT = max(fw_ckpts, key=os.path.getmtime)
print('FineWeb-pretrained checkpoint:', FW_CKPT)

RUN_FW_SFT = f'{OUT_ROOT}/runs_fineweb_smoltalk_sft'
!python opera-chat/train_chat.py \
  --data $DATA_FULL \
  --init-weights-from $FW_CKPT \
  --steps 10000 \
  --batch $FW_BATCH \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1664 --nb 416 --num-layers 8 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --compile default \
  --out-dir $RUN_FW_SFT \
  --save-every 1000 \
  --resume

FineWeb-pretrained checkpoint: /content/drive/MyDrive/opera-lm-demo/runs_fineweb/opera_v8_0_pe-none_data-chat_rotfree_amp_foreach_cmp-default_gpudata_aux0.25_gb2.0,0.0,-2.0_cur64x250_tie_opt-muon_msup.pt
data: train=223987 short=11692 long=44614 vocab=16384
=== OPERA-LM v9.0 (Spinor Tree, tree-training line) [pe-none+data-chat+rotfree+amp+foreach+cmp-default+gpudata+aux0.25+gb2.0,0.0,-2.0+cur64x250+tie+opt-muon+msup] ===
  steps=10000, batch=16, train max_len=256, eval_max_len=2048
  d=1664, nb=416, num_layers=8, lock=none, device=cuda
  pe=none, fold=left, fold_rotors=shared, fold_scale=False
  norm=layer, act=tanh, node_residual=False
  rot=free, seed=42, data=chat
  out=/content/drive/MyDrive/opera-lm-demo/runs_fineweb_smoltalk_sft, save_every=1000, resume=True
  tie=True, dropout=0.0, msup=True (weight 0.1)
  OPT: compile=default, gpu_data=True, aux_frac=0.25, amp=True, foreach=True
  References (train<=20, 4L): OPERA pe-none 70.92 / 1.65x / 1.68x; RoPE transformer 70.71 / 1.60x / 

### Quick check, then publish Stage 2 as its own Space
Same pattern as sections 6-7, pointed at `RUN_FW_SFT` instead, and a distinct model/Space repo name (`-v2` suffix) so it doesn't overwrite Stage 1's -- you end up with two separately-published demos to compare.

In [ ]:
os.environ['MODEL_DIR'] = RUN_FW_SFT
cfg_path = f'{RUN_FW_SFT}/model_config.json'
assert os.path.exists(cfg_path), f'missing {cfg_path}'
%cd opera-chat
!MODEL_DIR=$RUN_FW_SFT python -c "import app; app.demo.launch(share=True)"
%cd ..

In [ ]:
fw_sft_ckpts = [f for f in glob.glob(f'{RUN_FW_SFT}/*.pt') if not f.endswith('_train_ckpt.pt')]
FW_SFT_CKPT = max(fw_sft_ckpts, key=os.path.getmtime)
print('Stage 2 checkpoint:', FW_SFT_CKPT)

MODEL_REPO_V2 = f'{HF_USERNAME}/opera-lm-chat-v2'
SPACE_REPO_V2 = f'{HF_USERNAME}/opera-lm-chat-v2-space'

!python opera-chat/push_to_hub.py \
  --ckpt $FW_SFT_CKPT \
  --config $RUN_FW_SFT/model_config.json \
  --tokenizer opera-chat/tokenizer.json \
  --model-repo $MODEL_REPO_V2 \
  --space-repo $SPACE_REPO_V2

### Matched transformer baseline on the same FineWeb-Edu data
Same reasoning as section 8: the pretrain-only PPL above (general web text) isn't comparable to anything from Stage 1 (smoltalk) -- it's only meaningful against a model trained on the *same* `DATA_FINEWEB` pkl. Same matched config as section 8 (d=1264, nheads=8, num_layers=7 -> 155.05M params vs OPERA's 154.8M), same steps as the FineWeb pretrain phase above (20000, not the SFT phase's 10000) so both are compared at the same point.

In [ ]:
!python opera-chat/train_tf_chat.py \
  --pe rope \
  --data $DATA_FINEWEB \
  --steps 30 \
  --batch 16 \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1264 --num-layers 7 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --out-dir /content/smoke_tf_fw \
  --save-every 0

[baseline 2026-07-19-t3] opera_transformer_baseline_v2.py
data: train=97560 short=5411 long=23526 vocab=16384
=== transformer baseline [rope] on chat protocol ===
  params: 155,049,777 (tied)  steps=30 batch=16 max_len=256 device=cuda
  Muon: 134,206,464 matrix params (lr 0.02) + AdamW: 20,843,313 (lr 0.001)
  Initial per-token perplexity: 24756229665726.38 (chance ~ 16384)
  step     0  loss 30.3687  lr 0.00000  (0.4s, 0.36s/step)  T_cur 64
  step    29  loss 26.8898  lr 0.00006  (2.1s, 0.07s/step)  T_cur 64

=== Final: [rope] in-length PPL 620733701021.10 (5k) ===
  len 257-512: PPL 723266324653.78 (n=5797, 1.17x in-length)
  len 513-768: PPL 790473515648.35 (n=4699, 1.27x in-length)
  len 769-1024: PPL 823394139403.92 (n=3510, 1.33x in-length)
  len 1025-1280: PPL 840895876903.56 (n=2537, 1.35x in-length)
  len 1281-1536: PPL 857250018157.41 (n=2011, 1.38x in-length)
  len 1537-1792: PPL 870250998341.16 (n=1897, 1.40x in-length)
  len 1793-2048: PPL 876349657575.10 (n=3075, 1.41x in

In [ ]:
TF_FW_BATCH = 16  # set from the smoke test above
RUN_TF_FW = f'{OUT_ROOT}/runs_fineweb_tf'
!python opera-chat/train_tf_chat.py \
  --pe rope \
  --data $DATA_FINEWEB \
  --steps 20000 \
  --batch $TF_FW_BATCH \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1264 --num-layers 7 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --out-dir $RUN_TF_FW \
  --save-every 1000 \
  --resume

[baseline 2026-07-19-t3] opera_transformer_baseline_v2.py
data: train=97560 short=5411 long=23526 vocab=16384
=== transformer baseline [rope] on chat protocol ===
  params: 155,049,777 (tied)  steps=20000 batch=16 max_len=256 device=cuda
  Muon: 134,206,464 matrix params (lr 0.02) + AdamW: 20,843,313 (lr 0.001)
  Initial per-token perplexity: 24756229665726.38 (chance ~ 16384)
  step     0  loss 30.3687  lr 0.00000  (0.3s, 0.33s/step)  T_cur 64
  step   200  loss 7.3279  lr 0.00040  (11.4s, 0.06s/step)  T_cur 64
  step   400  loss 6.4013  lr 0.00080  (23.5s, 0.06s/step)  T_cur 128
  step   600  loss 5.9671  lr 0.00100  (37.9s, 0.06s/step)  T_cur 256
  step   800  loss 5.7270  lr 0.00100  (54.2s, 0.07s/step)  T_cur 256
  step  1000  loss 5.7774  lr 0.00100  (70.4s, 0.07s/step)  T_cur 256
    checkpoint -> /content/drive/MyDrive/opera-lm-demo/runs_fineweb_tf/tf_rope_opt-muon_train_ckpt.pt
    per-token perplexity: 347.27
  step  1200  loss 5.5277  lr 0.00100  (89.8s, 0.07s/step)  T_cur 2

### Matched transformer: same fine-tune phase, for the full comparison
Completes the symmetry: without this, the FineWeb comparison only covers the pretrain checkpoint, and whether OPERA's edge (or lack thereof) survives into the actual demo-quality end state stays untested -- exactly the kind of gap that undermines a comparison later. `--init-weights-from` was just added to `train_tf_chat.py` for this (it never had it before -- that script doesn't go through `opera_lm.train.train()`, so this flag needed wiring up separately, same pattern as OPERA's version). Same `--steps 10000` as OPERA's SFT phase above.

In [ ]:
tf_fw_ckpts = [f for f in glob.glob(f'{RUN_TF_FW}/*.pt') if not f.endswith('_train_ckpt.pt')]
TF_FW_CKPT = max(tf_fw_ckpts, key=os.path.getmtime)
print('transformer FineWeb-pretrained checkpoint:', TF_FW_CKPT)

RUN_TF_FW_SFT = f'{OUT_ROOT}/runs_fineweb_smoltalk_sft_tf'
!python opera-chat/train_tf_chat.py \
  --pe rope \
  --data $DATA_FULL \
  --init-weights-from $TF_FW_CKPT \
  --steps 10000 \
  --batch $TF_FW_BATCH \
  --max-len 256 \
  --eval-max-len 2048 \
  --d 1264 --num-layers 7 \
  --device cuda \
  --optimizer muon --muon-lr 0.02 \
  --out-dir $RUN_TF_FW_SFT \
  --save-every 1000 \
  --resume

transformer FineWeb-pretrained checkpoint: /content/drive/MyDrive/opera-lm-demo/runs_fineweb_tf/tf_rope_opt-muon.pt
[baseline 2026-07-19-t3] opera_transformer_baseline_v2.py
data: train=223987 short=11692 long=44614 vocab=16384
  Initialized weights from /content/drive/MyDrive/opera-lm-demo/runs_fineweb_tf/tf_rope_opt-muon.pt
=== transformer baseline [rope] on chat protocol ===
  params: 155,049,777 (tied)  steps=10000 batch=16 max_len=256 device=cuda
  Muon: 134,206,464 matrix params (lr 0.02) + AdamW: 20,843,313 (lr 0.001)
  Initial per-token perplexity: 266.49 (chance ~ 16384)
  step     0  loss 5.8298  lr 0.00000  (0.3s, 0.33s/step)  T_cur 64
  step   200  loss 3.4084  lr 0.00040  (11.3s, 0.06s/step)  T_cur 64
  step   400  loss 3.2320  lr 0.00080  (23.3s, 0.06s/step)  T_cur 128
  step   600  loss 3.7260  lr 0.00100  (37.7s, 0.06s/step)  T_cur 256
  step   800  loss 3.0514  lr 0.00100  (53.9s, 0.07s/step)  T_cur 256
  step  1000  loss 3.2603  lr 0.00099  (70.1s, 0.07s/step)  T_cur 